In [0]:
%pip install -q databricks-openai
dbutils.library.restartPython()

In [0]:
from databricks.sdk import WorkspaceClient
from databricks_openai import DatabricksOpenAI

w = WorkspaceClient()

client = DatabricksOpenAI(
    workspace_client=w,
    base_url=f"{w.config.host}/ai-gateway/mlflow/v1"
)

print("Embedding client ready.")

In [0]:
embedding_response = client.embeddings.create(
    model="system.ai.bge-large-en",
    input="Customers may request a return within 30 calendar days of delivery."
)

embedding = embedding_response.data[0].embedding

print("Vector dimension:", len(embedding))
print("First 10 values:")
print(embedding[:10])

In [0]:
texts = [
    "Customers have 30 days after delivery to return a normal item.",
    "What is the standard window for sending a purchase back?",
    "A monthly cancellation rate above 15% requires priority review.",
    "The warehouse ships products to customers."
]

response = client.embeddings.create(
    model="system.ai.bge-large-en",
    input=texts
)

vectors = [
    item.embedding
    for item in response.data
]

print("Vectors generated:", len(vectors))
print("Dimension:", len(vectors[0]))

In [0]:
import numpy as np

def cosine_similarity(a, b):
    a = np.array(a)
    b = np.array(b)

    return np.dot(a, b) / (
        np.linalg.norm(a) * np.linalg.norm(b)
    )

In [0]:
for i in range(1, len(texts)):
    score = cosine_similarity(
        vectors[0],
        vectors[i]
    )

    print(
        f"\n{texts[0]}\n"
        f"vs\n{texts[i]}\n"
        f"Similarity: {score:.4f}"
    )

In [0]:
chunks = (
    spark.table(
        "workspace.eacc_ecommerce_rag.chunks"
    )
    .select(
        "chunk_id",
        "document_title",
        "section_title",
        "chunk_text"
    )
    .limit(10)
    .collect()
)